In [ ]:
from sympy import *
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import dblquad
# Constants
pi = np.pi
G = 6.67e-11 #m^3/kg/s^2
M = 7.735e22 #kg
R = 1.737e6 #m
S = pi*R**2 #m^2

# Parameters for standard model (no meteor showers, no ice)
F_imp0 = 11.6e-15 #kg/m^2/s
Y0 = 1000
u_min0 = 130 #m/s
u_max0 = 4800 #m/s
v_imp0 = 20000 #m/s

Y_ice = 10000

# Parameters
alpha = 0.91
mmin = 3e-16 #kg
mmax = 1e-8 #kg
mu = 3.4

psi0 = pi/6 #radians

# Setting up variables
m, v, r, theta, phi = symbols('m v r theta phi', real=True, positive=True)

# Defining equation for initial velocity
h = r-R
u = sqrt(v**2 + 2*G*M*(1/R - 1/r))

# Defining equation for minimum initial velocity at certain altitude
u0 = sqrt(2*G*M*(1/R - 1/r))

# Defining equation for distribution of initial velocities
fu = ((mu-1)/(u0**(1-mu)))*(u**(-mu))

# Defining equation for ejection cone angle
psi = asin(((r*v) / (R*u)) * sin(theta))

# Defining equation for distribution of ejection angles
fpsi = (sin(psi))/(1-cos(psi0))

# Defining the distribution of the rate of grains produced
mdist = (1-alpha)/(mmax**(1-alpha)-mmin**(1-alpha)) * m**(-alpha-1) 
Nplus = integrate(mdist, m)
Nplus = Nplus.subs(m, mmax) - Nplus.subs(m, mmin)

# Phase space density of dust above surface
n = Nplus / (8*pi*R*r) * (fu*fpsi) / (v*u**2*sin(theta)*cos(psi))
n = n*2*pi # Integrating over phi

nFunc = lambdify((v, r, theta), n)

In [ ]:
def calculateThetaMax(r_val):
    umax = 4800
    vmax = sqrt(umax**2+2*G*M*(1/r_val - 1/R))
    ratio = sin(psi0)*R*umax/(r_val*vmax)
    thetamax = asin(ratio)
    return thetamax

def calculateVelocities(v_imp, Y):
    umin = symbols('umin', real=True, positive=True)

    K = 0.2
    ratio = 130/4800

    eqn = Eq(Y * (mu-1)/(3-mu) * (umin/v_imp)**2 * ((ratio)**(mu-3)-1), K)
    umin_shower = nsolve(eqn, umin, 500)
    umax_shower = umin_shower/ratio

    return umin_shower, umax_shower


def calculateDensity(r_val, umin, umax, F_imp, Y):
    Mplus = F_imp * Y * S # kg/s

    # Calculating the maximum and minimum velocities at a given altitude
    vmax = sqrt(umax**2+2*G*M*(1/r_val - 1/R))
    # Absolute minimum initial velocity is 130ms^-1 so anything below 4900m altitude will have it
    vmin = sqrt(umin**2+2*G*M*(1/r_val - 1/R)) # min u for low altitude is 130
    vmid = sqrt(2400**2+2*G*M*(1/r_val - 1/R)) # max u of not reaching escape velocity

    # Integrating the phase space density over the velocity and angle to get the density at a given altitude
    density1, error1 = dblquad(lambda v, theta: nFunc(v, r_val, theta), vmin, vmid, 0, calculateThetaMax(r_val))
    density2, error2 = dblquad(lambda v, theta: nFunc(v, r_val, theta), vmid, vmax, 0, calculateThetaMax(r_val))
    density = 2*density1 + density2

    return density*2*Mplus # theta in opposite direction, symmetry



In [ ]:
x1 = np.linspace(1, 100, 100)

print("Calculating base model (no ice) density of dust grains at low altitude above the Moon's surface...")
y0 = [calculateDensity(h+R, u_min0, u_max0, F_imp0, Y0)/1e4 for h in x1] #standard model (no meteor showers, no ice caps)

ice = int(input("What percentage of the surface is ice compared to regolith? (0-100) "))
print("Calculating density of dust grains at low altitude above the Moon's surface with " + str(ice) + "% ice caps...")
y_normal = [((100-ice)/100 * n) for n in y0]
umin_ice, umax_ice = calculateVelocities(v_imp0, Y_ice)
y_ice = [ice/100 * calculateDensity(h+R, umin_ice, umax_ice, F_imp0, Y_ice)/1e4 for h in x1]
y1 = np.add(y_normal, y_ice)

meteor = input("Is there a meteor shower? (y/n) ")
if meteor == 'y':
    level = input("What level of meteor shower? (1-4) ")
    if level == '1':
        v_imp = 40000
        F_imp = 1e-17
    elif level == '2':
        v_imp = 70000
        F_imp = 1e-17
    elif level == '3':
        v_imp = 40000
        F_imp = 5e-16
    elif level == '4':
        v_imp = 70000
        F_imp = 5e-16
    print("Calculating density of dust grains due to meteor shower...")
    umin_shower, umax_shower = calculateVelocities(v_imp, Y0)
    umin_shower_ice, umax_shower_ice = calculateVelocities(v_imp, Y_ice)
    y_meteor_noIce = [(100-ice)/100 * calculateDensity(h+R, umin_shower, umax_shower, F_imp, Y0)/1e4 for h in x1]
    y_meteor_ice = [ice/100 * calculateDensity(h+R, umin_shower_ice, umax_shower_ice, F_imp, Y_ice)/1e4 for h in x1]
    y_meteor = np.add(y_meteor_noIce, y_meteor_ice)
    y2 = np.add(y_meteor, y1)
    plt.plot(x1, y2, 'k--', x1, y1, 'b', x1, y0, 'r--')
    plt.legend(['Meteor Shower', 'No Meteor Shower', 'Base model'])
else:
    plt.plot(x1, y1, 'b', x1, y0, 'r--')
    plt.legend(['Ice', 'Base model'])        

plt.xlabel('Altitude (m)')
plt.ylabel('Density (x$\\mathrm{10}^{4}$ $\\mathrm{m}^{-3}$)')
plt.xlim(0,100)
plt.ylim(0,10)
plt.grid(True)

plt.show()